In [16]:
# Import the tools we'll use in our sales analysis.
import csv
from pathlib import Path

print("My sales analysis notebook is ready!")

My sales analysis notebook is ready!


In [17]:
from pathlib import Path

# Point directly to your project folder on the Desktop.
project_folder = Path.home() / "Desktop" / "day2-sales-analytics"

# Locate the CSV file inside the data folder.
data_path = project_folder / "data" / "electronics_sales.csv"

print("CSV file exists:", data_path.exists())
print("CSV path:", data_path)


CSV file exists: True
CSV path: C:\Users\PC\Desktop\day2-sales-analytics\data\electronics_sales.csv


Loading the csv

In [18]:
import csv

def read_sales(path):
    # Open the CSV and read each row as a dictionary.
    with open(path, mode="r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        columns = reader.fieldnames

    print("Number of rows:", len(rows))
    print("Column names:", columns)

    return rows

sales = read_sales(data_path)

Number of rows: 50
Column names: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


Creating the parse_number() function

In [19]:
def parse_number(value):
    # Reject missing or empty values.
    if value is None or value.strip() == "":
        raise ValueError("Value is missing")

    # Remove commas and the KES currency label.
    value = value.replace(",", "").replace("KES", "").strip()

    try:
        return float(value)
    except ValueError:
        raise ValueError("This value is not a valid number")

Testing the parse_number(value)

In [21]:
print(parse_number("45000"))
print(parse_number("45,000"))
print(parse_number("KES 45,000"))

45000.0
45000.0
45000.0


Cleaning sales records

In [22]:
def convert_rows(rows):
    clean_rows = []  #sales records that are clean
    skipped_rows = []  # sales records with errors

    for row in rows:
        try:

            row["quantity"] = parse_number(row["quantity"]) # converting quantity into numbers
            row["unit_price_kes"] = parse_number(row["unit_price_kes"]) # Converting unit_price into numbers


            if row["quantity"] < 0 or row["unit_price_kes"] < 0:  # Rejecting negative quanities and prices.
                raise ValueError("Quantity or price cannot be negative")

            clean_rows.append(row)

        except ValueError as error:
            skipped_rows.append({
                "sale_id": row.get("sale_id"),
                "reason": str(error)
            })

    print("Valid sales:", len(clean_rows))
    print("Skipped sales:", len(skipped_rows))
    print("Skippped rows:", skipped_rows)

    return clean_rows, skipped_rows






In [23]:
clean_sales, skipped_sales = convert_rows(sales)

Valid sales: 46
Skipped sales: 4
Skippped rows: [{'sale_id': 'S-3006', 'reason': 'This value is not a valid number'}, {'sale_id': 'S-3030', 'reason': 'This value is not a valid number'}, {'sale_id': 'S-3035', 'reason': 'Value is missing'}, {'sale_id': 'S-3039', 'reason': 'Value is missing'}]


Calculating Total revenue 

In [26]:
def total_revenue(rows):
    total = 0

    for row in rows:

        revenue = row["quantity"] * row["unit_price_kes"]

        total = total + revenue

    return total

Calculating revenue by store

In [35]:
def revenue_by (rows, field):
    totals = {}

    for row in rows:

        name = row[field] # getting product name or store from the current record

        revenue = row["quantity"] * row["unit_price_kes"] # Calculating revenue for this sale

        if name in totals:
            totals[name] = totals[name] + revenue

        else:
            totals[name] = revenue

    return totals





In [36]:
store_revenue = revenue_by(clean_sales, "store")
print(store_revenue)

{'Westgate': 235979.0, 'Kisumu Mega Plaza': 449678.0, 'Nakuru Westside': 139988.0, 'Mombasa City Mall': 154082.0, 'Nairobi CBD': 331085.0}


Creating top_n() function 

In [37]:
def top_n (totals, n):

    sorted_totals = sorted(
        totals.items(),
        key = lambda item: item[1],
        reverse = True
    )

    return sorted_totals[:n]

In [39]:
print("Top 3 stores")
print(top_n(store_revenue, 3))

Top 3 stores
[('Kisumu Mega Plaza', 449678.0), ('Nairobi CBD', 331085.0), ('Westgate', 235979.0)]


In [38]:
product_revenue = revenue_by(clean_sales, "product")

print("Top 5 products:")
print(top_n(product_revenue, 5))


Top 5 products:
[('HP 15 Laptop', 389994.0), ('Hisense 43in Smart TV', 272993.0), ('Lenovo Tab M10', 199992.0), ('Tecno Spark 20', 161988.0), ('Infinix Hot 40', 63996.0)]


Finding high value sales

In [41]:
def high_value(rows, threshold):
    return [
        row for row in rows
        if row["quantity"] * row["unit_price_kes"] >= threshold
    ]

In [42]:
high_value_sales = high_value(clean_sales, 50000)

print("Number of high value sales:", len(high_value_sales))
print(high_value_sales)

Number of high value sales: 5
[{'sale_id': 'S-3013', 'sale_date': '2026-09-08', 'store': 'Westgate', 'product': 'Lenovo Tab M10', 'quantity': 4.0, 'unit_price_kes': 24999.0, 'payment_method': 'M-Pesa'}, {'sale_id': 'S-3017', 'sale_date': '2026-09-10', 'store': 'Kisumu Mega Plaza', 'product': 'Hisense 43in Smart TV', 'quantity': 4.0, 'unit_price_kes': 38999.0, 'payment_method': 'M-Pesa'}, {'sale_id': 'S-3022', 'sale_date': '2026-09-13', 'store': 'Nairobi CBD', 'product': 'HP 15 Laptop', 'quantity': 4.0, 'unit_price_kes': 64999.0, 'payment_method': 'M-Pesa'}, {'sale_id': 'S-3036', 'sale_date': '2026-09-21', 'store': 'Nakuru Westside', 'product': 'Tecno Spark 20', 'quantity': 4.0, 'unit_price_kes': 13499.0, 'payment_method': 'M-Pesa'}, {'sale_id': 'S-3048', 'sale_date': '2026-09-30', 'store': 'Kisumu Mega Plaza', 'product': 'HP 15 Laptop', 'quantity': 2.0, 'unit_price_kes': 64999.0, 'payment_method': 'Card'}]


In [45]:
high_value_by_store = {}

for row in high_value_sales:
    store = row["store"]

    if store in high_value_by_store:
        high_value_by_store[store] += 1
    else:
        high_value_by_store[store] = 1

print(high_value_by_store)

{'Westgate': 1, 'Kisumu Mega Plaza': 2, 'Nairobi CBD': 1, 'Nakuru Westside': 1}


Creating reusable statistical functions

In [46]:
def calculate_mean (numbers):
    if len (numbers) == 0:
        raise ValueError("Cannot calculate the mean of an empty list")

    total = 0

    for number in numbers:
        total = total + number

    mean = total / len(numbers)
    return mean 


In [47]:
def calculate_median(numbers):
    if len(numbers) == 0:
        raise ValueError ("Cannot calculate the median of an empty list")

    sorted_numbers = sorted(numbers)

    middle = len(sorted_numbers) // 2

    if len(sorted_numbers) % 2 == 1:
        return sorted_numbers[middle]

    else:
        left = sorted_numbers[middle -1]
        right = sorted_numbers[middle]

        return (left + right) / 2


In [53]:
def minimum_maximum(numbers):
    if len(numbers) == 0:
        raise ValueError("Empty list, cannot be analysed")

    smallest = numbers[0]
    largest = numbers[0]

    for number in numbers:
        if number < smallest:
            smallest = number 

        if number > largest:
            largest = number

    return smallest, largest



In [54]:
# Test 

sales = [45000, 12000, 80000, 25000, 60000]

smallest, largest = minimum_maximum(sales)

print("Minimum sale:", smallest)
print("Maximum sale:", largest)

Minimum sale: 12000
Maximum sale: 80000


In [55]:
from pathlib import Path

project_folder = Path.home() / "Desktop" / "day2-sales-analytics"
output_folder = project_folder / "output"

# Create the output folder if it doesn't exist.
output_folder.mkdir(parents=True, exist_ok=True)

report_path = output_folder / "sales_report.txt"

print("Report will be saved to:", report_path)

Report will be saved to: C:\Users\PC\Desktop\day2-sales-analytics\output\sales_report.txt


In [71]:
# Calculate the total revenue from all valid sales.
total = total_revenue(clean_sales)

print("Total revenue: KES", total)

Total revenue: KES 1310812.0


In [58]:
store_revenue = revenue_by(clean_sales, "store")
product_revenue = revenue_by(clean_sales, "product")

In [59]:
with open(report_path, "w", encoding="utf-8") as file:
    file.write("TECH HUB KENYA SALES REPORT\n")
    file.write("===========================\n\n")

    file.write(f"Total revenue: KES {total:,.2f}\n\n")

    file.write("Revenue by store:\n")
    for store, revenue in store_revenue.items():
        file.write(f"{store}: KES {revenue:,.2f}\n")

    file.write("\nRevenue by product:\n")
    for product, revenue in product_revenue.items():
        file.write(f"{product}: KES {revenue:,.2f}\n")

print("Sales report saved successfully!")

Sales report saved successfully!


Verifying results with pandas

In [61]:
import pandas as pd

df = pd.read_csv(data_path)

print(df.head())

  sale_id   sale_date              store             product quantity  \
0  S-3001  2026-09-01           Westgate  Safaricom Neon Ray        1   
1  S-3002  2026-09-02           Westgate   USB-C Charger 25W        2   
2  S-3003  2026-09-02  Kisumu Mega Plaza      Lenovo Tab M10        2   
3  S-3004  2026-09-02           Westgate   Xiaomi Redmi Buds        2   
4  S-3005  2026-09-03    Nakuru Westside      Tecno Spark 20        2   

  unit_price_kes payment_method  
0           5999           Cash  
1           1499         M-Pesa  
2          24999           Cash  
3           3199         M-Pesa  
4          13499         M-Pesa  


In [69]:
# Convert quantity and price to numeric values.
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price_kes"] = pd.to_numeric(df["unit_price_kes"], errors="coerce")

# Keep rows with valid, non-negative quantity and price.
df = df.dropna(subset=["quantity", "unit_price_kes"])
df = df[
    (df["quantity"] >= 0) &
    (df["unit_price_kes"] >= 0)
]

# Calculate revenue for each sale.
df["revenue"] = df["quantity"] * df["unit_price_kes"]

pandas_total = df["revenue"].sum()

print("Pandas total revenue: KES", pandas_total)

Pandas total revenue: KES 1310812.0


In [70]:
if abs(total - pandas_total) < 0.01:
    print("MATCH: Both methods give the same total revenue.")
else:
    print("MISMATCH: Check the cleaning rules and calculations.")

MATCH: Both methods give the same total revenue.


Comparing revenue by store using pandas

In [72]:
# Add up the revenue from each store.
pandas_store_revenue = df.groupby("store")["revenue"].sum()

print(pandas_store_revenue)

store
Kisumu Mega Plaza    449678.0
Mombasa City Mall    154082.0
Nairobi CBD          331085.0
Nakuru Westside      139988.0
Westgate             235979.0
Name: revenue, dtype: float64


In [73]:
print("Standard Python results:")
print(store_revenue)

print("\nPandas results:")
print(pandas_store_revenue.to_dict())

Standard Python results:
{'Westgate': 235979.0, 'Kisumu Mega Plaza': 449678.0, 'Nakuru Westside': 139988.0, 'Mombasa City Mall': 154082.0, 'Nairobi CBD': 331085.0}

Pandas results:
{'Kisumu Mega Plaza': 449678.0, 'Mombasa City Mall': 154082.0, 'Nairobi CBD': 331085.0, 'Nakuru Westside': 139988.0, 'Westgate': 235979.0}


verifying revenue by product

In [74]:
print("Standard Python results:")
print(store_revenue)

print("\nPandas results:")
print(pandas_store_revenue.to_dict())

Standard Python results:
{'Westgate': 235979.0, 'Kisumu Mega Plaza': 449678.0, 'Nakuru Westside': 139988.0, 'Mombasa City Mall': 154082.0, 'Nairobi CBD': 331085.0}

Pandas results:
{'Kisumu Mega Plaza': 449678.0, 'Mombasa City Mall': 154082.0, 'Nairobi CBD': 331085.0, 'Nakuru Westside': 139988.0, 'Westgate': 235979.0}


Verifying high-value sales (python vs pandas)

In [75]:
# Keep sales whose revenue is at least KES 50,000.
pandas_high_value = df[df["revenue"] >= 50000]

print("High-value sales:", len(pandas_high_value))
print(pandas_high_value[["sale_id", "store", "product", "revenue"]])

High-value sales: 5
   sale_id              store                product   revenue
12  S-3013           Westgate         Lenovo Tab M10   99996.0
16  S-3017  Kisumu Mega Plaza  Hisense 43in Smart TV  155996.0
21  S-3022        Nairobi CBD           HP 15 Laptop  259996.0
35  S-3036    Nakuru Westside         Tecno Spark 20   53996.0
47  S-3048  Kisumu Mega Plaza           HP 15 Laptop  129998.0


In [76]:
print("Standard Python high-value sales:", len(high_value_sales))
print("Pandas high-value sales:", len(pandas_high_value))

if len(high_value_sales) == len(pandas_high_value):
    print("MATCH: The number of high-value sales is the same.")
else:
    print("MISMATCH: Check the cleaning rules and threshold.")

Standard Python high-value sales: 5
Pandas high-value sales: 5
MATCH: The number of high-value sales is the same.


Verifying High value sales

In [77]:
python_high_value_ids = []

for row in high_value_sales:
    python_high_value_ids.append(row["sale_id"])

print(python_high_value_ids)

['S-3013', 'S-3017', 'S-3022', 'S-3036', 'S-3048']


In [78]:
pandas_high_value_ids = pandas_high_value["sale_id"].tolist()

print(pandas_high_value_ids)

['S-3013', 'S-3017', 'S-3022', 'S-3036', 'S-3048']


In [79]:
if set(python_high_value_ids) == set(pandas_high_value_ids):
    print("MATCH: Both methods identified the same sales.")
else:
    print("MISMATCH: The methods identified different sales.")

MATCH: Both methods identified the same sales.


Verifying high value sales by store

In [80]:
pandas_high_value_by_store = pandas_high_value["store"].value_counts()

print(pandas_high_value_by_store)

store
Kisumu Mega Plaza    2
Westgate             1
Nairobi CBD          1
Nakuru Westside      1
Name: count, dtype: int64


In [81]:
print("Standard Python:")
print(high_value_by_store)

print("\nPandas:")
print(pandas_high_value_by_store.to_dict())

Standard Python:
{'Westgate': 1, 'Kisumu Mega Plaza': 2, 'Nairobi CBD': 1, 'Nakuru Westside': 1}

Pandas:
{'Kisumu Mega Plaza': 2, 'Westgate': 1, 'Nairobi CBD': 1, 'Nakuru Westside': 1}


In [82]:
python_counts = high_value_by_store
pandas_counts = pandas_high_value_by_store.to_dict()

all_stores = set(python_counts) | set(pandas_counts)

match = True

for store in all_stores:
    python_count = python_counts.get(store, 0)
    pandas_count = pandas_counts.get(store, 0)

    if python_count != pandas_count:
        match = False
        print("Mismatch for:", store)

if match:
    print("MATCH: High-value sales counts agree for every store.")

MATCH: High-value sales counts agree for every store.
